# 03 — Steam Genre and Platform Analysis

## Objective

This notebook examines how videogame characteristics such as **genre and platform availability** relate to representation, player reception, popularity, and market positioning on Steam.

The previous notebook established the overall market context. This analysis moves to the product level to identify patterns that may help Ubisoft evaluate the positioning of a future Steam release.

The analysis focuses on the following business questions:

1. Which genres are most represented on Steam?
2. Which genres show the strongest player reception?
3. Which genres attract the greatest levels of player attention and commercial-reach proxies?
4. How does pricing vary across major genres?
5. How common is Windows, macOS, and Linux support?
6. How prevalent is cross-platform availability?
7. How do genre and platform characteristics interact?
8. Which product characteristics appear associated with stronger popularity or reception indicators?

Because individual games can belong to multiple genres, genre-level analysis requires transforming the game-level genre arrays into separate game-genre observations. Spark's `explode()` operation is used for this transformation.

Care is taken to distinguish the number of **game-genre associations** from the number of **unique games**, using distinct application identifiers where necessary to avoid unintended double-counting.

As in the previous analysis, review volume, concurrent users, and ownership estimates are interpreted as popularity or engagement proxies rather than direct measures of sales or profitability.

## Methodology

The analysis remains **Spark-first**. Filtering, array transformations, aggregations, percentile calculations, and cross-dimensional comparisons are performed using PySpark DataFrame operations.

Genre analysis uses an exploded representation derived from the prepared game-level dataset. The original game-level DataFrame remains unchanged and is retained for analyses where one row per videogame is required.

Visualizations are produced from aggregated Spark results in Databricks. Only analytically useful charts are retained, with emphasis on results that support clear business interpretation rather than maximizing the number of visualizations.

## 1. Environment and Prepared Data Loading

The validated game-level dataset created during data preparation is loaded from the persisted Delta table.

Two analytical granularities will be used in this notebook:

- the original **game-level dataset**, containing one row per Steam videogame;
- an **exploded game-genre dataset**, containing one row per game-genre association.

Maintaining these two levels separately prevents accidental double-counting when calculating metrics that require unique games.

In [0]:
# PySpark imports
# ---------------------------------------------------------------------------

from pyspark.sql import functions as F


# Load the prepared game-level dataset
# ---------------------------------------------------------------------------

TABLE_NAME = "steam_games_prepared"

steam_games_df = spark.table(TABLE_NAME)

print(f"Steam videogames: {steam_games_df.count():,}")
print(f"Analytical columns: {len(steam_games_df.columns)}")

Steam videogames: 55,690
Analytical columns: 37


## 2. Genre Data Transformation

Steam games can be associated with multiple genres. In the prepared dataset, these genres are stored as an array for each game.

To analyze individual genres, the array is transformed using Spark's `explode()` operation. Each element of the genre array becomes a separate row while retaining the corresponding game identifier and analytical variables.

For example, a game associated with both `Action` and `Adventure` contributes one observation to each of those genre categories.

This transformation changes the analytical granularity from **one row per game** to **one row per game-genre association**. Consequently, genre-level game counts use `countDistinct(appid)` whenever the objective is to measure unique videogames.

In [0]:
# Transform genre arrays into individual game-genre observations
# ---------------------------------------------------------------------------

game_genres_df = (
    steam_games_df
    .filter(F.col("genres").isNotNull())
    .select(
        "appid",
        "name",
        "publisher",
        "release_year",
        "initial_price",
        "is_free",
        "total_reviews",
        "positive_review_ratio",
        "concurrent_users",
        "owner_midpoint",
        "supports_windows",
        "supports_mac",
        "supports_linux",
        "platform_count",
        F.explode("genres").alias("genre")
    )
    .filter(
        F.col("genre").isNotNull()
        & (F.trim(F.col("genre")) != "")
    )
)

In [0]:
# Validate the exploded genre dataset
# ---------------------------------------------------------------------------

genre_transformation_summary_df = (
    game_genres_df
    .agg(
        F.count("*").alias("game_genre_associations"),
        F.countDistinct("appid").alias("unique_games"),
        F.countDistinct("genre").alias("distinct_genres")
    )
)

display(genre_transformation_summary_df)

game_genre_associations,unique_games,distinct_genres
157110,55530,28


In [0]:
# Inspect genre labels and their representation
# ---------------------------------------------------------------------------

genre_counts_df = (
    game_genres_df
    .groupBy("genre")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .orderBy(F.desc("games"))
)

display(genre_counts_df)

genre,games
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


### 2. Genre Transformation Validation

The genre transformation produces **157,110 game-genre associations** from **55,530 unique videogames** with available genre information.

This difference is expected because individual games can belong to multiple genres. The exploded dataset therefore contains approximately 2.8 genre associations per represented game and must not be interpreted as containing 157,110 unique videogames.

The source taxonomy contains **28 distinct genre labels**. The most frequently represented labels include Indie, Action, Casual, Adventure, Strategy, Simulation, and RPG.

The source also classifies a small number of videogame applications under broader labels such as Utilities, Design & Illustration, Education, and Movie. These labels are retained rather than manually reclassified because the analysis aims to preserve the taxonomy provided by the source dataset.

All subsequent genre-level game counts use distinct Steam application identifiers where appropriate to prevent unintended double-counting.

## 3. Genre Representation

Genre representation provides a high-level view of the types of products competing for visibility on Steam.

Because games can belong to multiple genres, genre counts are not mutually exclusive and their percentages should not be expected to sum to 100%. A game classified as both Action and Adventure contributes to both categories.

In [0]:
# Calculate representation of the major Steam genres
# ---------------------------------------------------------------------------

GAMES_WITH_GENRE = game_genres_df.select("appid").distinct().count()

top_genres_df = (
    genre_counts_df
    .withColumn(
        "percentage_of_games",
        F.round(
            F.col("games") / F.lit(GAMES_WITH_GENRE) * 100,
            2
        )
    )
    .limit(12)
)

display(top_genres_df)

genre,games,percentage_of_games
Indie,39681,71.46
Action,23759,42.79
Casual,22086,39.77
Adventure,21431,38.59
Strategy,10895,19.62
Simulation,10836,19.51
RPG,9534,17.17
Early Access,6145,11.07
Free to Play,3393,6.11
Sports,2666,4.8


In [0]:
# Prepare major genre representation for visualization
# ---------------------------------------------------------------------------

top_genres_chart_df = (
    top_genres_df
    .select(
        "genre",
        "games"
    )
)

display(top_genres_chart_df)

genre,games
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

### 3. Key Findings — Genre Representation

Genre representation on Steam is strongly concentrated around a small number of broad and overlapping categories.

**Indie** is the most frequently represented genre label, associated with **39,681 games (71.46% of games with genre information)**. It is followed by **Action with 23,759 games (42.79%)**, **Casual with 22,086 (39.77%)**, and **Adventure with 21,431 (38.59%)**.

A second group of major genres includes Strategy (19.62%), Simulation (19.51%), and RPG (17.17%). Early Access is associated with 11.07% of games, while Free to Play represents 6.11%.

More specialized categories such as Sports, Racing, and Massively Multiplayer account for smaller proportions of the catalogue.

These percentages are **not mutually exclusive** because individual games can belong to several genres simultaneously. They therefore measure the proportion of games associated with each genre label rather than exclusive market shares.

For Ubisoft, the results indicate that broad categories such as Action and Adventure operate within particularly crowded segments of the Steam catalogue. Genre representation alone does not reveal whether these categories generate stronger player reception or popularity, so the next analysis compares genre presence with performance indicators.

## 4. Player Reception by Genre

Genre representation describes the supply side of the Steam catalogue, but a frequently represented genre is not necessarily associated with stronger player reception.

To make reception comparisons more robust, this analysis reuses the minimum threshold of **750 total reviews** established in the market analysis. Games below this threshold are excluded from the genre-level reception comparison.

Because games can belong to multiple genres, the resulting metrics describe games **associated with each genre** rather than mutually exclusive genre groups.

### 4.1 Reception Among Established Games

In [0]:
# Compare player reception across genres with substantial review evidence
# ---------------------------------------------------------------------------

MIN_REVIEWS = 750

genre_reception_df = (
    game_genres_df
    .filter(F.col("total_reviews") >= MIN_REVIEWS)
    .groupBy("genre")
    .agg(
        F.countDistinct("appid").alias("qualifying_games"),
        F.round(
            F.expr(
                "percentile_approx(positive_review_ratio, 0.5)"
            ) * 100,
            2
        ).alias("median_positive_review_pct"),
        F.round(
            F.avg("positive_review_ratio") * 100,
            2
        ).alias("average_positive_review_pct")
    )
    .orderBy(
        F.desc("median_positive_review_pct"),
        F.desc("qualifying_games")
    )
)

display(genre_reception_df)

genre,qualifying_games,median_positive_review_pct,average_positive_review_pct
Photo Editing,7,94.53,89.38
Animation & Modeling,24,90.38,86.62
Casual,1393,88.37,84.99
Design & Illustration,23,87.29,86.56
Indie,3565,87.23,83.89
Adventure,2470,86.71,83.26
Web Publishing,12,86.51,88.41
Game Development,7,86.51,88.57
Gore,7,86.37,87.12
Utilities,44,86.26,84.75


Some of those 28 labels may have only a handful of games with ≥750 reviews. We shouldn't call a genre “best-performing” based on few titles. So I create the decision-oriented version with a minimum of 100 qualifying games.

750 reviews per game protects against unstable game-level ratings, while 100 qualifying games per genre protects against drawing genre-level conclusions from tiny groups.

In [0]:
# Retain genres with sufficient representation for comparison
# ---------------------------------------------------------------------------

MIN_QUALIFYING_GAMES = 100

established_genre_reception_df = (
    genre_reception_df
    .filter(
        F.col("qualifying_games") >= MIN_QUALIFYING_GAMES
    )
    .orderBy(
        F.desc("median_positive_review_pct"),
        F.desc("qualifying_games")
    )
)

display(established_genre_reception_df)

genre,qualifying_games,median_positive_review_pct,average_positive_review_pct
Casual,1393,88.37,84.99
Indie,3565,87.23,83.89
Adventure,2470,86.71,83.26
Simulation,1495,84.63,80.84
Action,3024,84.59,81.05
RPG,1526,84.02,80.82
Early Access,484,83.52,79.45
Strategy,1463,83.47,80.34
Racing,246,82.73,80.41
Sports,275,81.28,78.13


In [0]:
# Prepare established genre reception for visualization
# ---------------------------------------------------------------------------

genre_reception_chart_df = (
    established_genre_reception_df
    .select(
        "genre",
        "median_positive_review_pct"
    )
    .orderBy(F.desc("median_positive_review_pct"))
)

display(genre_reception_chart_df)

genre,median_positive_review_pct
Casual,88.37
Indie,87.23
Adventure,86.71
Simulation,84.63
Action,84.59
RPG,84.02
Early Access,83.52
Strategy,83.47
Racing,82.73
Sports,81.28


Databricks visualization. Run in Databricks to view.

### 4. Key Findings — Player Reception by Genre

After requiring at least **750 reviews per game** and at least **100 qualifying games per genre**, 12 sufficiently represented genre labels remain for comparison.

Among these established genres, **Casual** has the highest median positive-review ratio at **88.37%**, followed by **Indie (87.23%)** and **Adventure (86.71%)**.

Simulation, Action, RPG, Early Access, and Strategy form a middle group, with median positive-review ratios between approximately **83% and 85%**. Racing and Sports are slightly lower, while Free to Play records a median of **80.74%**.

**Massively Multiplayer** stands out with the lowest median positive-review ratio among the sufficiently represented genres, at **71.72%**, despite having 404 games that meet the 750-review threshold.

The results demonstrate that catalogue representation and player reception are distinct dimensions. Action, for example, is one of the most prevalent genre labels on Steam but does not lead the reception ranking. Conversely, Casual games show the strongest median reception among the established genre groups.

These comparisons describe associations rather than causal effects. Genre labels overlap, and differences in reception may also reflect factors such as game quality, audience expectations, monetization model, release lifecycle, or other product characteristics not isolated by this analysis.

## 5. Popularity and Commercial-Reach Proxies by Genre

Positive reviews measure player reception but do not indicate how much attention a game attracts. Genre-level popularity is therefore examined separately using three complementary indicators:

- **total review volume**, representing cumulative player interaction with the Steam review system;
- **estimated ownership midpoint**, providing an approximate indicator of commercial reach;
- **concurrent users**, providing a snapshot indicator of active player engagement.

These variables have strongly skewed distributions. Median values are therefore used to compare typical games within each genre rather than allowing a small number of blockbuster titles to dominate genre averages.

Ownership remains a range-derived approximation, while concurrent users represent activity at a particular observation point. Neither metric is interpreted as exact sales or lifetime commercial performance.

In [0]:
# Compare typical popularity and reach across sufficiently represented genres
# ---------------------------------------------------------------------------

MIN_GENRE_GAMES = 100

genre_popularity_df = (
    game_genres_df
    .groupBy("genre")
    .agg(
        F.countDistinct("appid").alias("games"),
        F.expr(
            "percentile_approx(total_reviews, 0.5)"
        ).alias("median_total_reviews"),
        F.expr(
            "percentile_approx(owner_midpoint, 0.5)"
        ).alias("median_owner_midpoint"),
        F.expr(
            "percentile_approx(concurrent_users, 0.5)"
        ).alias("median_concurrent_users")
    )
    .filter(F.col("games") >= MIN_GENRE_GAMES)
    .orderBy(F.desc("median_total_reviews"))
)

display(genre_popularity_df)

genre,games,median_total_reviews,median_owner_midpoint,median_concurrent_users
Free to Play,3393,133,35000.0,0
Massively Multiplayer,1460,124,35000.0,0
RPG,9534,43,10000.0,0
Simulation,10836,36,10000.0,0
Strategy,10895,33,10000.0,0
Violent,168,32,10000.0,0
Adventure,21431,30,10000.0,0
Animation & Modeling,322,27,10000.0,0
Racing,2155,26,10000.0,0
Sports,2666,25,10000.0,0


In [0]:
# Compare popularity across the same established genres used for reception
# ---------------------------------------------------------------------------

established_genres = [
    row["genre"]
    for row in established_genre_reception_df
        .select("genre")
        .collect()
]

established_genre_popularity_df = (
    genre_popularity_df
    .filter(F.col("genre").isin(established_genres))
    .orderBy(F.desc("median_total_reviews"))
)

display(established_genre_popularity_df)

genre,games,median_total_reviews,median_owner_midpoint,median_concurrent_users
Free to Play,3393,133,35000.0,0
Massively Multiplayer,1460,124,35000.0,0
RPG,9534,43,10000.0,0
Simulation,10836,36,10000.0,0
Strategy,10895,33,10000.0,0
Adventure,21431,30,10000.0,0
Racing,2155,26,10000.0,0
Action,23759,25,10000.0,0
Sports,2666,25,10000.0,0
Indie,39681,24,10000.0,0


In [0]:
# Prepare genre popularity for visualization
# ---------------------------------------------------------------------------

genre_popularity_chart_df = (
    established_genre_popularity_df
    .select(
        "genre",
        "median_total_reviews"
    )
    .orderBy(F.desc("median_total_reviews"))
)

display(genre_popularity_chart_df)

genre,median_total_reviews
Free to Play,133
Massively Multiplayer,124
RPG,43
Simulation,36
Strategy,33
Adventure,30
Racing,26
Action,25
Sports,25
Indie,24


Databricks visualization. Run in Databricks to view.

### 5. Key Findings — Popularity by Genre

Typical player attention varies substantially across Steam genre labels.

**Free to Play** records the highest median review volume, with **133 reviews per game**, followed closely by **Massively Multiplayer with 124**. These values are considerably higher than those of the remaining sufficiently represented genres.

RPG ranks next with a median of **43 reviews**, followed by Simulation (36), Strategy (33), and Adventure (30). Large catalogue categories such as Action and Indie record substantially lower typical review volumes, with medians of **25 and 24 reviews** respectively.

This provides an important contrast with the reception analysis. In particular, **Massively Multiplayer combines relatively high typical review activity with the lowest median positive-review ratio among the established genre groups (71.72%)**. Free to Play similarly shows comparatively high review volume but a lower median positive-review ratio of 80.74%.

These patterns reinforce the distinction between **player attention and player satisfaction**. Genres associated with greater review activity are not necessarily those receiving the most positive reception.

The ownership and concurrent-user medians provide less differentiation at genre level. Most genres share a median ownership midpoint of approximately 10,000, reflecting the broad ownership ranges provided by the source, while the median concurrent-user count is zero across all sufficiently represented genres. These variables are therefore retained as supporting popularity indicators but are not used as the primary genre-level comparison.

## 6. Pricing Across Genres

Genre positioning may also differ in terms of pricing strategy.

To compare typical prices without allowing free-to-play titles to mechanically reduce genre medians, this analysis considers **paid games only**. Median initial price is used because the market-level analysis showed that Steam prices are strongly right-skewed.

The results describe observed catalogue pricing patterns and should not be interpreted as optimal prices for a future Ubisoft release.

In [0]:
# Compare initial prices across established genres for paid games
# ---------------------------------------------------------------------------

genre_price_df = (
    game_genres_df
    .filter(
        (F.col("is_free") == False)
        & F.col("initial_price").isNotNull()
        & F.col("genre").isin(established_genres)
    )
    .groupBy("genre")
    .agg(
        F.countDistinct("appid").alias("paid_games"),
        F.round(
            F.expr("percentile_approx(initial_price, 0.5)"),
            2
        ).alias("median_initial_price"),
        F.round(
            F.avg("initial_price"),
            2
        ).alias("average_initial_price")
    )
    .orderBy(F.desc("median_initial_price"))
)

display(genre_price_df)

genre,paid_games,median_initial_price,average_initial_price
Early Access,5046,9.99,10.81
Simulation,9451,8.99,10.76
Massively Multiplayer,686,8.99,11.08
RPG,8140,7.99,10.97
Strategy,9386,7.99,10.06
Sports,2253,7.99,10.96
Adventure,19019,6.99,9.36
Action,20582,5.99,9.21
Racing,1882,5.99,9.7
Indie,34765,4.99,7.76


### 6. Key Findings — Pricing Across Genres

Pricing differs meaningfully across the established Steam genre groups.

Among paid games, **Early Access** has the highest median initial price at **9.99**, followed by **Simulation** and **Massively Multiplayer** at **8.99**. RPG, Strategy, and Sports each have a median initial price of **7.99**.

Adventure occupies the middle of the distribution with a median of **6.99**, while Action and Racing both record **5.99**. The large Indie and Casual categories have lower median initial prices of **4.99**.

Across every established genre, the average price exceeds the median price. This is consistent with the right-skewed pricing distribution identified in the market-level analysis, where a smaller number of relatively expensive games raise the average above the price of a typical title.

The `Free to Play` genre label contains a small number of games classified as paid according to their initial price. This is not necessarily a data error: `Free to Play` is a source-provided genre classification, whereas the project's `is_free` feature is derived independently from the recorded price. The two variables therefore represent different concepts.

For Ubisoft, these results provide contextual benchmarks rather than a direct pricing recommendation. Genre is associated with different catalogue price levels, but pricing decisions should also consider product scope, positioning, monetization model, and competitive set.

## 7. Platform Availability

Platform support affects the potential accessibility and reach of a videogame across the Steam ecosystem.

The dataset identifies availability on three desktop operating systems:

- Windows
- macOS
- Linux

Because a single game may support multiple operating systems, individual platform percentages are not mutually exclusive.

This section first measures overall platform availability and then examines the prevalence of single-platform and multi-platform releases.

In [0]:
# Summarize support for each operating system
# ---------------------------------------------------------------------------

platform_support_df = (
    steam_games_df
    .agg(
        F.countDistinct("appid").alias("total_games"),
        F.sum(
            F.when(F.col("supports_windows"), 1).otherwise(0)
        ).alias("windows_games"),
        F.sum(
            F.when(F.col("supports_mac"), 1).otherwise(0)
        ).alias("mac_games"),
        F.sum(
            F.when(F.col("supports_linux"), 1).otherwise(0)
        ).alias("linux_games")
    )
)

display(platform_support_df)

total_games,windows_games,mac_games,linux_games
55690,55675,12769,8457


In [0]:
# Calculate platform availability percentages
# ---------------------------------------------------------------------------

total_games = steam_games_df.count()

platform_availability_df = (
    steam_games_df
    .select(
        F.explode(
            F.array(
                F.struct(
                    F.lit("Windows").alias("platform"),
                    F.col("supports_windows").cast("int").alias("supported")
                ),
                F.struct(
                    F.lit("macOS").alias("platform"),
                    F.col("supports_mac").cast("int").alias("supported")
                ),
                F.struct(
                    F.lit("Linux").alias("platform"),
                    F.col("supports_linux").cast("int").alias("supported")
                )
            )
        ).alias("platform_data")
    )
    .select("platform_data.*")
    .groupBy("platform")
    .agg(
        F.sum("supported").alias("games")
    )
    .withColumn(
        "percentage_of_games",
        F.round(F.col("games") / F.lit(total_games) * 100, 2)
    )
    .orderBy(F.desc("games"))
)

display(platform_availability_df)

platform,games,percentage_of_games
Windows,55675,99.97
macOS,12769,22.93
Linux,8457,15.19


### 7.2 Cross-Platform Availability

In [0]:
# Examine the number of supported operating systems per game
# ---------------------------------------------------------------------------

platform_count_distribution_df = (
    steam_games_df
    .groupBy("platform_count")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .withColumn(
        "percentage_of_games",
        F.round(F.col("games") / F.lit(total_games) * 100, 2)
    )
    .orderBy("platform_count")
)

display(platform_count_distribution_df)

platform_count,games,percentage_of_games
1,41285,74.13
2,7599,13.65
3,6806,12.22


In [0]:
# Identify the most common platform combinations
# ---------------------------------------------------------------------------

platform_combinations_df = (
    steam_games_df
    .withColumn(
        "platform_combination",
        F.concat_ws(
            " + ",
            F.when(F.col("supports_windows"), F.lit("Windows")),
            F.when(F.col("supports_mac"), F.lit("macOS")),
            F.when(F.col("supports_linux"), F.lit("Linux"))
        )
    )
    .groupBy("platform_combination")
    .agg(
        F.countDistinct("appid").alias("games")
    )
    .withColumn(
        "percentage_of_games",
        F.round(F.col("games") / F.lit(total_games) * 100, 2)
    )
    .orderBy(F.desc("games"))
)

display(platform_combinations_df)

platform_combination,games,percentage_of_games
Windows,41271,74.11
Windows + macOS + Linux,6806,12.22
Windows + macOS,5951,10.69
Windows + Linux,1647,2.96
macOS,11,0.02
Linux,3,0.01
macOS + Linux,1,0.0


In [0]:
# Prepare platform combinations for visualization
# ---------------------------------------------------------------------------

platform_combinations_chart_df = (
    platform_combinations_df
    .filter(F.col("games") >= 10)
    .select(
        "platform_combination",
        "percentage_of_games"
    )
    .orderBy(F.desc("percentage_of_games"))
)

display(platform_combinations_chart_df)

platform_combination,percentage_of_games
Windows,74.11
Windows + macOS + Linux,12.22
Windows + macOS,10.69
Windows + Linux,2.96
macOS,0.02


Databricks visualization. Run in Databricks to view.

### 7.2 Key Findings — Platform Availability

Steam videogame availability is overwhelmingly concentrated on **Windows**.

Windows is supported by **55,675 games (99.97% of the catalogue)**, compared with **12,769 games (22.93%)** supporting macOS and **8,457 (15.19%)** supporting Linux. Because games may support multiple operating systems, these percentages are not mutually exclusive.

The platform-count distribution shows that **74.13% of games support only one operating system**, while 13.65% support two and just **12.22% support all three platforms**.

The platform-combination analysis confirms that this single-platform concentration is almost entirely driven by Windows. **41,271 games (74.11%) are Windows-only**, making it by far the dominant deployment pattern. The next most common configurations are Windows + macOS + Linux (12.22%), Windows + macOS (10.69%), and Windows + Linux (2.96%).

Only a negligible number of games exclude Windows entirely, confirming that Windows functions as the effective baseline platform within this Steam dataset.

For Ubisoft, the results suggest that Windows support provides access to virtually the entire Steam catalogue context, while macOS and Linux support represent additional cross-platform reach rather than standard market coverage. However, these catalogue frequencies alone do not establish whether supporting additional operating systems improves player reception or popularity.

## 8. Genre and Platform Availability

Overall platform statistics show that Windows dominates the Steam catalogue, but platform strategy may vary across genres.

This section combines the exploded genre representation with platform characteristics to measure how frequently games associated with each established genre support macOS, Linux, or all three operating systems.

Because genre labels overlap, the results describe platform patterns among games associated with each genre rather than mutually exclusive market segments.

In [0]:
# Compare platform availability across established genres
# ---------------------------------------------------------------------------

genre_platform_df = (
    game_genres_df
    .filter(F.col("genre").isin(established_genres))
    .groupBy("genre")
    .agg(
        F.countDistinct("appid").alias("games"),
        F.round(
            F.avg(F.col("supports_windows").cast("int")) * 100,
            2
        ).alias("windows_pct"),
        F.round(
            F.avg(F.col("supports_mac").cast("int")) * 100,
            2
        ).alias("mac_pct"),
        F.round(
            F.avg(F.col("supports_linux").cast("int")) * 100,
            2
        ).alias("linux_pct"),
        F.round(
            F.avg(
                F.when(F.col("platform_count") == 3, 1).otherwise(0)
            ) * 100,
            2
        ).alias("all_three_pct")
    )
    .orderBy(F.desc("all_three_pct"))
)

display(genre_platform_df)

genre,games,windows_pct,mac_pct,linux_pct,all_three_pct
Indie,39681,99.99,25.04,17.59,14.16
Strategy,10895,99.97,27.58,16.76,13.91
RPG,9534,99.99,23.58,15.98,13.58
Adventure,21431,99.98,23.51,15.41,12.78
Casual,22086,99.98,23.23,14.96,12.0
Simulation,10836,99.96,22.51,14.14,11.71
Free to Play,3393,99.94,24.9,13.97,11.32
Action,23759,99.98,19.21,14.22,10.94
Racing,2155,99.95,19.68,14.11,10.58
Massively Multiplayer,1460,99.93,18.49,11.23,9.18


In [0]:
# Compare reception and popularity by number of supported platforms
# ---------------------------------------------------------------------------

platform_performance_df = (
    steam_games_df
    .groupBy("platform_count")
    .agg(
        F.countDistinct("appid").alias("games"),
        F.expr(
            "percentile_approx(total_reviews, 0.5)"
        ).alias("median_total_reviews"),
        F.round(
            F.expr(
                """
                percentile_approx(
                    CASE
                        WHEN total_reviews >= 750
                        THEN positive_review_ratio
                    END,
                    0.5
                )
                """
            ) * 100,
            2
        ).alias("median_positive_review_pct_750"),
        F.expr(
            "percentile_approx(owner_midpoint, 0.5)"
        ).alias("median_owner_midpoint"),
        F.expr(
            "percentile_approx(concurrent_users, 0.5)"
        ).alias("median_concurrent_users")
    )
    .orderBy("platform_count")
)

display(platform_performance_df)

platform_count,games,median_total_reviews,median_positive_review_pct_750,median_owner_midpoint,median_concurrent_users
1,41285,21,84.17,10000.0,0
2,7599,37,87.59,10000.0,0
3,6806,79,88.55,10000.0,0


In [0]:
# Prepare review engagement by platform breadth for visualization
# ---------------------------------------------------------------------------

platform_engagement_chart_df = (
    platform_performance_df
    .select(
        "platform_count",
        "median_total_reviews"
    )
    .orderBy("platform_count")
)

display(platform_engagement_chart_df)

platform_count,median_total_reviews
1,21
2,37
3,79


Databricks visualization. Run in Databricks to view.

### 8. Key Findings — Genre and Platform Availability

Cross-platform availability varies across established Steam genre groups, although Windows support remains effectively universal across all of them.

**Indie** games show the highest share of three-platform support, with **14.16%** available on Windows, macOS, and Linux. Strategy (13.91%), RPG (13.58%), and Adventure (12.78%) also show comparatively broad cross-platform availability.

At the other end of the distribution, three-platform support is less common among Massively Multiplayer (9.18%), Sports (8.66%), and particularly Early Access games (7.18%).

macOS support shows a similar pattern, reaching 27.58% among Strategy games and 25.04% among Indie games, while falling to 14.65% among Early Access titles. Linux availability remains lower across all established genres.

These differences suggest that cross-platform deployment practices vary by product category. However, the analysis describes catalogue associations and does not establish that genre itself determines platform strategy.

### 8. Platform Breadth and Game Performance

Games supporting a broader range of operating systems also show stronger observed engagement and reception indicators in this dataset.

Games supporting only **one platform** have a median of **21 reviews**, compared with **37 reviews for two-platform games** and **79 reviews for games supporting all three operating systems**.

Among games meeting the established 750-review threshold, median positive-review reception also increases with platform breadth: from **84.17% for single-platform games**, to **87.59% for two-platform games**, and **88.55% for three-platform games**.

The relationship is not reproduced across every popularity proxy. Median ownership midpoint remains approximately **10,000** for all three groups, while median concurrent-user activity remains zero. These variables provide limited discrimination because ownership is recorded in broad ranges and concurrent activity is highly concentrated.

The results therefore indicate an **association**, rather than a causal effect, between broader platform availability and stronger review-based performance. More successful or established games may be more likely to receive additional platform support, while cross-platform availability may itself expand potential audience reach. The observational dataset does not allow these explanations to be separated.

For Ubisoft, cross-platform support can therefore be considered a potential reach and accessibility dimension, but these results do not demonstrate that adding macOS or Linux support would independently cause higher player engagement or reception.

## 9. Focused Multivariate Analysis

The previous sections examined market characteristics individually and across selected categorical dimensions. A final focused multivariate analysis evaluates linear relationships between a small set of numerical game characteristics and performance indicators.

The analysis considers:

- initial price;
- total review volume;
- positive-review ratio;
- estimated ownership midpoint;
- concurrent users;
- number of supported languages;
- number of supported operating systems;
- release year.

Pearson correlation coefficients are used as descriptive indicators of **linear association**. They do not establish causal relationships, and weak correlations do not rule out nonlinear or category-specific relationships.

Because review volume, ownership, and concurrent-user activity are highly skewed, the resulting coefficients should be interpreted cautiously and alongside the distributional findings established earlier.

In [0]:
# Define focused numerical variables for correlation analysis
# ---------------------------------------------------------------------------

correlation_columns = [
    "initial_price",
    "total_reviews",
    "positive_review_ratio",
    "owner_midpoint",
    "concurrent_users",
    "language_count",
    "platform_count",
    "release_year"
]


# Calculate pairwise Pearson correlations using Spark
# ---------------------------------------------------------------------------

correlation_results = []

for variable_1 in correlation_columns:
    for variable_2 in correlation_columns:
        correlation_value = steam_games_df.stat.corr(
            variable_1,
            variable_2
        )

        correlation_results.append(
            (
                variable_1,
                variable_2,
                round(correlation_value, 3)
                if correlation_value is not None
                else None
            )
        )


# Convert the small aggregated result back to a Spark DataFrame
# ---------------------------------------------------------------------------

correlation_df = spark.createDataFrame(
    correlation_results,
    [
        "variable_1",
        "variable_2",
        "correlation"
    ]
)

display(correlation_df)

variable_1,variable_2,correlation
initial_price,initial_price,1.0
initial_price,total_reviews,0.042
initial_price,positive_review_ratio,0.061
initial_price,owner_midpoint,0.033
initial_price,concurrent_users,0.02
initial_price,language_count,0.127
initial_price,platform_count,0.005
initial_price,release_year,0.007
total_reviews,initial_price,0.042
total_reviews,total_reviews,1.0


In [0]:
# Retain unique variable pairs and rank the strongest relationships
# ---------------------------------------------------------------------------

correlation_pairs = []

for i, variable_1 in enumerate(correlation_columns):
    for variable_2 in correlation_columns[i + 1:]:

        correlation_value = steam_games_df.stat.corr(
            variable_1,
            variable_2
        )

        correlation_pairs.append(
            (
                variable_1,
                variable_2,
                round(correlation_value, 3)
                if correlation_value is not None
                else None
            )
        )


correlation_pairs_df = (
    spark.createDataFrame(
        correlation_pairs,
        [
            "variable_1",
            "variable_2",
            "correlation"
        ]
    )
    .withColumn(
        "absolute_correlation",
        F.round(F.abs("correlation"), 3)
    )
    .orderBy(F.desc("absolute_correlation"))
)

display(correlation_pairs_df)

variable_1,variable_2,correlation,absolute_correlation
total_reviews,concurrent_users,0.813,0.813
owner_midpoint,concurrent_users,0.773,0.773
total_reviews,owner_midpoint,0.578,0.578
initial_price,language_count,0.127,0.127
positive_review_ratio,platform_count,0.097,0.097
owner_midpoint,language_count,0.089,0.089
language_count,platform_count,0.089,0.089
total_reviews,language_count,0.085,0.085
initial_price,positive_review_ratio,0.061,0.061
concurrent_users,language_count,0.057,0.057


### 9. Key Findings — Multivariate Relationships

The focused correlation analysis reveals a clear distinction between **popularity and reach indicators** and most other game characteristics.

The strongest linear relationship is between **total review volume and concurrent users**, with a Pearson correlation of **0.813**. Estimated ownership midpoint is also strongly associated with concurrent users (**0.773**) and moderately associated with total review volume (**0.578**).

These relationships provide useful internal consistency across the project's popularity proxies: games attracting greater review activity also tend to appear in higher ownership ranges and show greater concurrent-player activity.

In contrast, **positive-review ratio has very weak linear relationships with the popularity indicators**. Its correlations with total reviews (0.024), ownership midpoint (0.025), and concurrent users (0.009) are all close to zero. This reinforces an important finding from the earlier analyses: **player satisfaction and player attention represent distinct dimensions of game performance**.

Most product characteristics also show weak linear relationships with the performance indicators. Initial price, language count, platform count, and release year have correlations close to zero with review volume, ownership, and concurrent users. The largest relationship outside the popularity indicators is only 0.127, between initial price and language count.

The earlier platform-group analysis showed higher median review volume and reception among games supporting more operating systems, while the game-level Pearson correlations involving platform count remain weak. These findings are not contradictory: aggregated group differences can exist even when the overall linear relationship across individual games is small.

These coefficients should be interpreted cautiously. Review volume, ownership, and concurrent-user activity are strongly right-skewed, and the ownership variable is derived from broad ranges. Pearson correlation measures linear association only and does not establish causality.

In [0]:
# Display the strongest numerical relationships
# ---------------------------------------------------------------------------

strongest_correlations_df = (
    correlation_pairs_df
    .filter(F.col("absolute_correlation") >= 0.10)
    .select(
        "variable_1",
        "variable_2",
        "correlation"
    )
)

display(strongest_correlations_df)

variable_1,variable_2,correlation
total_reviews,concurrent_users,0.813
owner_midpoint,concurrent_users,0.773
total_reviews,owner_midpoint,0.578
initial_price,language_count,0.127


## 10. Business Recommendations for Ubisoft

The Steam analysis provides several strategic considerations for Ubisoft when evaluating the positioning of a future videogame release.

### Compete on differentiation rather than catalogue volume

Steam has expanded substantially, with thousands of videogames released annually in recent years. The publisher landscape is also highly fragmented, with nearly 30,000 distinct publisher entries in the dataset.

Ubisoft already has a meaningful catalogue presence, ranking among the largest publishers by number of Steam games in this dataset. However, catalogue size alone does not indicate player reception or commercial success. In a crowded environment, product differentiation and player engagement are therefore more informative strategic considerations than release volume alone.

### Treat popularity and player satisfaction as separate objectives

The analyses consistently show that popularity and reception are distinct dimensions.

Massively Multiplayer and Free to Play games generate comparatively high typical review activity, yet their median positive-review ratios are below those of Casual, Indie, and Adventure games. At game level, positive-review ratio also shows almost no linear correlation with review volume, ownership midpoint, or concurrent users.

Ubisoft should therefore evaluate a future title using both **reach indicators** and **player-satisfaction indicators**, rather than treating either dimension as a complete measure of performance.

### Use genre benchmarks as context, not as deterministic targets

Action and Adventure are among the most represented labels on Steam, demonstrating substantial catalogue competition. Meanwhile, genre-level reception, popularity, and pricing vary considerably.

These results can help establish competitive benchmarks, but they do not identify a universally superior genre. Genre labels overlap, and observed performance also reflects product quality, audience expectations, monetization, lifecycle, and other factors not isolated by this dataset.

### Benchmark pricing against comparable product positioning

Most Steam videogames in the dataset are paid, while paid-game prices are right-skewed and differ across genre groups. Established genre medians range from approximately 4.99 for Indie and Casual games to 9.99 for Early Access games.

These values provide catalogue benchmarks rather than optimal price recommendations. Ubisoft should evaluate price alongside genre, production scope, monetization model, and intended market positioning.

### Consider localization as a reach strategy

English support is nearly universal, while broader multilingual support is concentrated in a smaller subset of games. The median game supports only one language despite an average of more than three.

For a large international publisher such as Ubisoft, localization can therefore remain an important accessibility and market-reach consideration, although this analysis does not establish that adding languages independently causes stronger commercial performance.

### Treat Windows as the baseline Steam platform

Windows support is effectively universal in the dataset, covering 99.97% of games. macOS and Linux availability are substantially less common, and only 12.22% of games support all three operating systems.

Games with broader platform support show higher median review activity and somewhat stronger reception in the descriptive analysis. However, game-level correlations with platform count are weak, and the observational data cannot determine whether broader platform support improves performance or whether already successful games are more likely to receive additional platform versions.

Windows should therefore be considered the baseline Steam platform, while macOS and Linux support should be evaluated through incremental audience potential relative to development and maintenance costs.

### Use existing Ubisoft titles as internal Steam benchmarks

Tom Clancy's Rainbow Six Siege is one of the most visible Ubisoft titles in the dataset, ranking among the most-reviewed Steam games with more than one million reviews and strong positive reception.

This provides a useful internal reference point for Ubisoft when comparing future releases, while recognizing that the performance of one established live-service title cannot be generalized to every future product.

## 11. Limitations

The analysis provides a broad descriptive view of the Steam videogame ecosystem, but several limitations affect the interpretation of the results.

- **The dataset is a snapshot.** Price, discounts, concurrent users, reviews, and other characteristics may change over time.
- **The latest release date is November 11, 2022.** Consequently, 2022 represents an incomplete year and should not be compared directly with complete previous years.
- **Review volume is a popularity proxy, not a sales metric.** Not every player writes a review, and reviewing behaviour may differ across games and audiences.
- **Positive-review ratio measures reception rather than commercial performance.** Highly rated games are not necessarily the most commercially successful.
- **Ownership is provided as broad ranges.** The midpoint used in this project is an analytical approximation and should not be interpreted as an exact ownership or sales estimate.
- **Concurrent users represent snapshot activity.** A value of zero does not imply that a game has never been played or has no historical audience.
- **Genre labels overlap.** A game can belong to multiple genres, so genre percentages and statistics represent associations rather than mutually exclusive market segments.
- **Publisher names are source-provided strings.** Variations in naming or multi-publisher entries were not manually consolidated into corporate groups.
- **Platform and language availability indicate presence only.** They do not measure implementation quality, localization depth, development cost, or platform-specific demand.
- **Correlation does not imply causation.** The observed relationships cannot determine whether particular product characteristics cause stronger game performance.
- **Several numerical variables are highly skewed.** Blockbuster games can influence averages and Pearson correlations, which is why median statistics were emphasized throughout the descriptive analysis.

These limitations mean that the findings should be interpreted as **market benchmarks and descriptive associations**, not forecasts of future Ubisoft sales or causal estimates of product-strategy effects.

## 12. Conclusion

This notebook extended the Steam market analysis by examining genre positioning, platform availability, and multivariate relationships.

The Steam catalogue is dominated by overlapping Indie, Action, Casual, and Adventure labels, but genre prevalence does not translate directly into stronger reception or popularity. Casual, Indie, and Adventure games show comparatively strong median player reception among sufficiently reviewed titles, while Free to Play and Massively Multiplayer games generate higher typical review activity but weaker reception.

Pricing also varies across genre groups, providing useful competitive benchmarks without identifying a single optimal pricing strategy.

Platform analysis confirms that Steam is overwhelmingly Windows-oriented. Cross-platform support remains comparatively uncommon, although games available on more operating systems show stronger review-based performance descriptively. This relationship remains associative rather than causal.

Finally, the multivariate analysis shows strong relationships between review volume, estimated ownership, and concurrent-player activity, while positive player reception remains largely independent of these popularity indicators.

Together with the market-level analysis, these findings provide Ubisoft with a structured view of Steam's competitive environment while demonstrating the importance of evaluating **market positioning, player reach, and player satisfaction as complementary rather than interchangeable dimensions**.